First we look at the process model 


In [48]:
import control as ct
import matplotlib.pyplot as plt
import numpy as np

S = .10 # 1e7 mu s / 1e9 ppb = 1e-2 

z = ct.tf('z')
G = S/(1-z**(-1))
plt.figure(1)
ct.pzmap(G)
plt.title("Pole/zero plot for plant")
# plt.figure(2)
# ct.bode_plot(G, display_margins=True)
# plt.title("Bode plot of plant")
# plt.show()


In [43]:
# hr = np.linspace(0, 50000, 10000)
# Gi = 1/( 1 - z**(-1))
# ct.rlocus(G*Gi , gains=hr)
# plt.title("Possible conjugate pairs of poles using I-controller")
# plt.show()


Now find the Kp and Ki using pole placement


In [63]:
import sympy as sp
import numpy as np

S = sp.symbols('S')
def auto_place_pi(desired_poles):
    
    z, Kp, Ki = sp.symbols('z Kp Ki')
    # Define the Transfer Function components
    plant_num = S * z
    plant_den = z - 1
    pi_num = Kp * (z - 1) + Ki * z
    pi_den = z - 1
    
    # Build the Characteristic Polynomial: (Plant_Den * PI_Den) + (Plant_Num * PI_Num) = 0
    char_eq = sp.Poly(plant_den * pi_den + plant_num * pi_num, z)
    actual_coeffs = char_eq.all_coeffs()
    # print("actual coeffs:")
    # sp.pprint(actual_coeffs)
    
    # Get the target polynomial coefficients from the desired poles
    desired_poly = np.poly(desired_poles)
    
    # Set up equations matching Actual coefficients to Desired coefficients
    # (We divide by actual_coeffs[0] to normalize the polynomial so the highest power is 1)
    equations = []
    for actual, desired in zip(actual_coeffs, desired_poly):
        equations.append(sp.Eq(actual / actual_coeffs[0], desired))
        
    # Let SymPy solve the algebra
    solution = sp.solve(equations, (Kp, Ki), dict=True)[0]
    return (solution[Kp]), (solution[Ki])




In [64]:
desired_poles = [0.3 + 0.1j, 0.3 - 0.1j]

Kp_repr, Ki_repr = auto_place_pi(desired_poles)

print(f"SymPy Calculated Kp: {Kp_repr}")
print(f"SymPy Calculated Ki: {Ki_repr}")


In [71]:
S_nom = 0.1

Kp_calc = float(Kp_repr.subs(S, S_nom))
Ki_calc = float(Ki_repr.subs(S, S_nom))

print(f"Calculated Kp: {Kp_calc:.4f}")
print(f"Calculated Ki: {Ki_calc:.4f}")


Now place the poles


In [66]:
Kp = 40
Ki = 50

def pzmap_with_ks(Kp, Ki):
    Gpi = (Kp*(z-1) + Ki*z)/(z-1)
    Gcl = ct.feedback(G*Gpi, 1)
    print(Gcl)
    poles = ct.poles(Gcl)
    plt.figure(1)
    ct.pzmap(Gcl)
    plt.title("Pole/zero plot of PI-controller")
    print(poles)
pzmap_with_ks(Kp, Ki)
# plt.figure(2)
# ct.bode_plot(G*Gpi, display_margins=True)
# plt.title("Bode plot of closed loop")
# plt.show()


In [67]:
def get_ks(desired_poles) -> tuple[float, float]:
    Kp_repr, Ki_repr = auto_place_pi(desired_poles)
    S_nom = 0.1
    
    Kp_calc = float(Kp_repr.subs(S, S_nom))
    Ki_calc = float(Ki_repr.subs(S, S_nom))
    return (Kp_calc, Ki_calc)


In [80]:
desired_poles = [0.3 + 0.3j, 0.3 - 0.3j]
(kp, ki) = get_ks(desired_poles)
print(f"{desired_poles=}\n\t{kp = :.2f}, {ki = :.2f}")

pzmap_with_ks(kp, ki)


In [84]:
desired_poles = [0.1 + 0.1j, 0.1 - 0.1j]

(kp, ki) = get_ks(desired_poles)

print(f"{desired_poles=}\n\t{kp = :.2f}, {ki = :.2f}")
pzmap_with_ks(kp, ki)
